# 15.02 · Pretrain a small LLM end to end

**Chapter:** Advanced & Frontier Topics · **Milestone:** M7

## Goals
- [ ] Run the whole pipeline yourself: **tokenizer → data pipeline → pretraining → evaluation → SFT → sampling**
- [ ] Train a BPE tokenizer and measure its compression
- [ ] Build a modern decoder (RMSNorm, RoPE, SwiGLU — what changed since the 04.03 GPT-2-style model)
- [ ] **Compute budgeting**: FLOPs ≈ 6·N·D, Chinchilla-optimal tokens, tokens/sec and MFU on your hardware
- [ ] Turn the base model into an instruction follower with a tiny SFT, and measure it

## Resources
- [nanochat](https://github.com/karpathy/nanochat) (full ChatGPT-style pipeline for ~$100 of GPU) · [nanoGPT](https://github.com/karpathy/nanoGPT) — free
- [Stanford CS336 – Language Modeling from Scratch](https://stanford-cs336.github.io/) — free
- [TinyStories paper](https://arxiv.org/abs/2305.07759) · [Chinchilla](https://arxiv.org/abs/2203.15556) — free
- [HF – The Smol Training Playbook](https://huggingface.co/spaces/HuggingFaceTB/smol-training-playbook) · [HF tokenizers](https://huggingface.co/docs/tokenizers) — free

Chapter 04 trained a character-level model on Shakespeare. This lesson does what real labs do, scaled down to ~10 minutes on a laptop:
**TinyStories** (short stories in simple English written by GPT-3.5/4) is the classic dataset where tiny models learn fluent text.

In [ ]:
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from torch import nn
from torch.nn import functional as F

from aieng import DATA

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)
WORK = DATA / "pretrain"
WORK.mkdir(exist_ok=True)
print(device)

## 1. Data
Stream the first 120k training stories (~90 MB of text) and 2k validation stories. At scale this step is most of the work: crawling,
deduplication, quality filtering, PII removal, mixing sources (see FineWeb, DCLM, Dolma).

In [ ]:
def stories(split, n):
    return [s["text"].strip() for s in load_dataset("roneneldan/TinyStories", split=split, streaming=True).take(n) if s["text"].strip()]


train_text, val_text = stories("train", 120_000), stories("validation", 2_000)
print(f"{len(train_text):,} train stories, {sum(map(len, train_text)) / 1e6:.0f}M characters\n---\n{train_text[0][:400]}")

## 2. Tokenizer: byte-level BPE
BPE starts from 256 bytes and repeatedly merges the most frequent adjacent pair into a new token (04.01). Small vocab = more tokens per text
(longer sequences); big vocab = bigger embedding matrix. Domain-specific tokenizers compress their domain much better than general ones.

In [ ]:
VOCAB = 4096
tok_path = WORK / f"bpe-{VOCAB}.json"
if not tok_path.exists():
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(vocab_size=VOCAB, special_tokens=["<|endoftext|>"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tokenizer.train_from_iterator(train_text, trainer)
    tokenizer.save(str(tok_path))
tokenizer = Tokenizer.from_file(str(tok_path))
EOT = tokenizer.token_to_id("<|endoftext|>")

from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained("gpt2")
sample = " ".join(val_text[:200])
print(f"ours  ({VOCAB:6} vocab): {len(sample) / len(tokenizer.encode(sample).ids):.2f} chars/token")
print(f"GPT-2 ({gpt2.vocab_size:6} vocab): {len(sample) / len(gpt2(sample).input_ids):.2f} chars/token")
print(tokenizer.encode("Once upon a time, Lily found a shiny red ball.").tokens)

A 4k-vocab tokenizer trained on the domain compresses it about as well as GPT-2's general 50k vocab — with a 12× smaller embedding table.

## 3. Data pipeline: tokenize once, pack into a flat binary file
Standard practice (nanoGPT, nanochat, Megatron): one long stream of token ids separated by `<|endoftext|>`, stored as `uint16`, memory-mapped,
and sampled as random windows. No padding, no wasted compute.

In [ ]:
def pack(texts, path):
    if not path.exists():
        ids = []
        for enc in tokenizer.encode_batch(texts):
            ids.extend(enc.ids); ids.append(EOT)
        np.array(ids, dtype=np.uint16).tofile(path)
    return np.memmap(path, dtype=np.uint16, mode="r")


train_ids, val_ids = pack(train_text, WORK / "train.bin"), pack(val_text, WORK / "val.bin")
print(f"train {len(train_ids) / 1e6:.1f}M tokens | val {len(val_ids) / 1e6:.2f}M tokens")

CTX, BATCH = 256, 32


def get_batch(data):
    ix = np.random.randint(0, len(data) - CTX - 1, BATCH)
    x = torch.from_numpy(np.stack([data[i: i + CTX] for i in ix]).astype(np.int64))
    y = torch.from_numpy(np.stack([data[i + 1: i + 1 + CTX] for i in ix]).astype(np.int64))
    return x.to(device), y.to(device)

## 4. A modern small decoder
Same skeleton as 04.03, with the upgrades every recent open model (Llama, Qwen, Gemma, OLMo) uses:
- **RMSNorm** instead of LayerNorm (cheaper, no mean-centering); pre-norm.
- **RoPE** (rotary position embeddings): rotate queries/keys by position instead of adding learned position vectors → better length handling.
- **SwiGLU** MLP: `W2(silu(W1 x) ⊙ W3 x)` instead of `W2(gelu(W1 x))`.
- No biases, tied input/output embeddings (small models), `scaled_dot_product_attention` (fused attention kernel).

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


def rope_cache(T, hd, base=10_000):
    freqs = 1.0 / base ** (torch.arange(0, hd, 2).float() / hd)
    angles = torch.outer(torch.arange(T).float(), freqs)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):  # x: (B, H, T, hd); rotate pairs (x1, x2) by position-dependent angles
    x1, x2 = x[..., ::2], x[..., 1::2]
    T = x.shape[2]
    c, s = cos[:T], sin[:T]
    return torch.stack([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).flatten(-2)


class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.n1, self.n2 = h, RMSNorm(d), RMSNorm(d)
        self.qkv, self.o = nn.Linear(d, 3 * d, bias=False), nn.Linear(d, d, bias=False)
        hidden = int(8 / 3 * d) // 64 * 64  # SwiGLU keeps params ≈ a 4d GELU MLP
        self.w1, self.w3, self.w2 = nn.Linear(d, hidden, bias=False), nn.Linear(d, hidden, bias=False), nn.Linear(hidden, d, bias=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, T, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        x = x + self.o(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, C))
        h = self.n2(x)
        return x + self.w2(F.silu(self.w1(h)) * self.w3(h))


class SmallLM(nn.Module):
    def __init__(self, vocab=VOCAB, d=384, layers=6, heads=6, ctx=CTX):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList(Block(d, heads) for _ in range(layers))
        self.norm = RMSNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.emb.weight
        cos, sin = rope_cache(ctx, d // heads)
        self.register_buffer("cos", cos, persistent=False); self.register_buffer("sin", sin, persistent=False)
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)

    def forward(self, idx, targets=None):
        x = self.emb(idx)
        for b in self.blocks:
            x = b(x, self.cos, self.sin)
        logits = self.head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, ids, n, temperature=0.8, top_k=50):
        for _ in range(n):
            logits = self(ids[:, -CTX:])[0][:, -1] / temperature
            logits[logits < torch.topk(logits, top_k).values[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(F.softmax(logits, -1), 1)
            ids = torch.cat([ids, nxt], 1)
            if nxt.item() == EOT:
                break
        return ids


model = SmallLM().to(device)
N = sum(p.numel() for p in model.parameters())
N_nonemb = N - model.emb.weight.numel()
print(f"{N / 1e6:.1f}M parameters ({N_nonemb / 1e6:.1f}M non-embedding)")

## 5. Compute budget
Training FLOPs ≈ **6 · N · D** (forward 2ND + backward 4ND; N = non-embedding params, D = tokens). Chinchilla: compute-optimal D ≈ **20 · N**.
Modern small models are trained far beyond that (Llama 3 8B: ~1,900 tokens/param; SmolLM2: ~11T tokens) because inference is cheaper
with a smaller, longer-trained model. First, measure what this laptop does per second:

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
for _ in range(3):  # warm-up
    loss = model(*get_batch(train_ids))[1]; loss.backward(); opt.zero_grad()
if device == "mps": torch.mps.synchronize()
t = time.perf_counter()
for _ in range(10):
    loss = model(*get_batch(train_ids))[1]; loss.backward(); opt.step(); opt.zero_grad()
if device == "mps": torch.mps.synchronize()
tps = 10 * BATCH * CTX / (time.perf_counter() - t)
flops = 6 * N_nonemb * tps
print(f"{tps:,.0f} tokens/s ≈ {flops / 1e12:.2f} TFLOP/s achieved")
TIME_BUDGET = 8 * 60  # seconds
D = tps * TIME_BUDGET
print(f"in {TIME_BUDGET // 60} min: ~{D / 1e6:.0f}M tokens = {D / N_nonemb:.0f} tokens/param (Chinchilla-optimal: 20) | {6 * N_nonemb * D:.1e} FLOPs")
for gpu, peak in [("M4 Pro GPU (fp32, ~rough)", 9e12), ("A100 (bf16)", 312e12), ("H100 (bf16)", 989e12)]:
    print(f"   {gpu:26} same run would take ≈ {6 * N_nonemb * D / (peak * 0.4) / 60:6.2f} min at 40% MFU")

**MFU** (model FLOPs utilisation) = achieved / peak FLOPs; well-tuned GPU training reaches 40–60%. Our tiny model on a laptop is far lower:
small matrices don't saturate the hardware. nanochat's d20 model (~560M params, ~11B tokens) is ~4 hours on 8×H100 (~$100).

## 6. Pretrain (time-boxed)
AdamW, linear warmup then cosine decay, gradient clipping. Validation loss every 250 steps.

In [ ]:
model = SmallLM().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95), weight_decay=0.1)
MAX_STEPS = int(D / (BATCH * CTX))
WARMUP, LR, MIN_LR = 200, 1e-3, 1e-4
lr_at = lambda s: LR * s / WARMUP if s < WARMUP else MIN_LR + (LR - MIN_LR) * 0.5 * (1 + math.cos(math.pi * min(1, (s - WARMUP) / (MAX_STEPS - WARMUP))))


@torch.no_grad()
def val_loss(iters=20):
    model.eval(); out = np.mean([model(*get_batch(val_ids))[1].item() for _ in range(iters)]); model.train()
    return out


history, start, step = [], time.perf_counter(), 0
while time.perf_counter() - start < TIME_BUDGET and step <= MAX_STEPS:
    for g in opt.param_groups: g["lr"] = lr_at(step)
    loss = model(*get_batch(train_ids))[1]
    opt.zero_grad(set_to_none=True); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    if step % 250 == 0:
        history.append((step, loss.item(), val_loss()))
        print(f"step {step:5} | train {history[-1][1]:.3f} | val {history[-1][2]:.3f} | {time.perf_counter() - start:4.0f}s")
    step += 1
history.append((step, loss.item(), val_loss()))
print(f"done: {step} steps, {step * BATCH * CTX / 1e6:.0f}M tokens, val loss {history[-1][2]:.3f} (perplexity {math.exp(history[-1][2]):.1f})")
torch.save(model.state_dict(), WORK / "base.pt")

s, tr, va = zip(*history)
plt.figure(figsize=(6, 3)); plt.plot(s, tr, label="train"); plt.plot(s, va, label="val"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()


def complete(prompt, n=150, m=model):
    m.eval()
    ids = torch.tensor([tokenizer.encode(prompt).ids], device=device)
    return tokenizer.decode(m.generate(ids, n)[0].tolist())


print(complete("Once upon a time, there was a little dog named"))

Train and val losses track each other: every batch is fresh data (we never repeat a token), so there's no overfitting — we are
**compute-bound**, not data-bound. At ~1 token per parameter we're far below Chinchilla's 20, so the loss is still falling: more compute
would buy a better model (exercise 2). A random-guess model starts at ln(4096) ≈ 8.3; the TinyStories paper's best small models reach ~1.2–1.5.

## 7. SFT: from "continue the text" to "follow an instruction"
A base model continues text. Supervised fine-tuning teaches a format: prompt → response. Our instruction: *write a story using three given
words*. We build examples from the stories themselves (pick 3 of the 400 most common content words that appear in a story → that story is the
response) and compute the loss **only on the response tokens** (labels = -100 on the prompt), exactly like 12.03.

Metric: the fraction of the 3 requested words that appear in the generated story, compared with a **control** — the same stories scored
against *another* prompt's words. Common words show up by chance, so only the gap above the control is instruction following.

In [ ]:
from collections import Counter

STOP = set("the a an and to was of in he she it his her they is on with for that had said you i one day there were very but".split())
counts = Counter(w.strip(".,!?\"'").lower() for s in train_text[:20_000] for w in s.split())
COMMON = set([w for w, _ in counts.most_common(800) if w.isalpha() and len(w) > 3 and w not in STOP][:400])


def sft_example(story, rng):
    words = sorted({w.strip(".,!?\"'").lower() for w in story.split()} & COMMON)
    if len(words) < 3:
        return None
    pick = rng.sample(words, 3)
    return f"Write a story using the words: {', '.join(pick)}.\nStory: ", story, pick


rng = random.Random(0)
sft_train = [e for e in (sft_example(s, rng) for s in train_text[:40_000]) if e]
sft_test = [e for e in (sft_example(s, rng) for s in val_text[:300]) if e][:40]


def sft_batch(examples):
    xs, ys = [], []
    for prompt, story, _ in examples:
        p, r = tokenizer.encode(prompt).ids, tokenizer.encode(story).ids + [EOT]
        ids = (p + r)[: CTX + 1]
        labels = ([-100] * len(p) + r)[: CTX + 1]
        pad = CTX + 1 - len(ids)
        xs.append(ids[:-1] + [EOT] * pad); ys.append(labels[1:] + [-100] * pad)
    return torch.tensor(xs, device=device), torch.tensor(ys, device=device)


def words_used(m):
    """(fraction of requested words used, control: fraction of ANOTHER prompt's words in the same story)."""
    used = control = 0
    for i, (prompt, _, words) in enumerate(sft_test):
        out = complete(prompt, 200, m)[len(prompt):].lower()
        used += sum(w in out for w in words) / 3
        control += sum(w in out for w in sft_test[(i + 1) % len(sft_test)][2]) / 3
    return used / len(sft_test), control / len(sft_test)


print("base model: words used {:.0%} (control {:.0%})".format(*words_used(model)))
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.0)
model.train()
start = time.perf_counter()
for step in range(1000):
    x, y = sft_batch(random.sample(sft_train, BATCH))
    loss = model(x, y)[1]
    opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
print(f"SFT: 1000 steps in {time.perf_counter() - start:.0f}s, loss {loss.item():.3f}")
print("SFT model : words used {:.0%} (control {:.0%})".format(*words_used(model)))
prompt, _, words = sft_test[0]
print("\n" + complete(prompt, 200))
torch.save(model.state_dict(), WORK / "sft.pt")

Read the gap to the control, not the raw number. The base model sits at the control (it ignores the instruction: it's just more text to
continue). After SFT the gap is small and varies run to run (a few points to ~15): the model learned the *format* ("Story:" → a story)
much more than the *conditioning*. Using words from the prompt requires copying — the induction heads of 15.04 — and a 10M model trained
for 8 minutes on 13M tokens barely has them. More pretraining (exercise 2) helps SFT more than more SFT steps do. The same recipe, scaled up, is how base models become assistants: SFT on (prompt, response) pairs in a chat template, then preference tuning
(12.04) and RL (12.05). A chat UI on top is the Gradio `ChatInterface` from 13.04.

## Exercises
1. Run a scaled-down **nanochat** on a rented GPU (paid: a few $ for small configs, e.g. `depth=8` on one A100/H100 for ~1 h): tokenizer →
   pretrain → midtrain → SFT → chat UI. Compare its loss curve and samples with this notebook.
2. Scaling experiment: train d=192/4 layers, d=384/6 layers and d=512/8 layers for the **same** time budget. Plot final val loss vs params.
   Which is compute-optimal on your laptop?
3. Ablate one modern component: LayerNorm instead of RMSNorm, learned positions instead of RoPE, or GELU instead of SwiGLU. Same budget, compare val loss.
4. Change `VOCAB` to 1024 and 16384. How do compression, tokens/sec and final quality per *character* (not per token!) change?

In [ ]:
# your code here